# Tokenization

Tokenization for previous maodel - chars (all the possible character in mini shakesphere dataset)

What are token to a LLM?

Token: a small chunk of text (a character, part of a word, or a whole word) that the model treats as a single unit, represented by an integer ID.

One of the popular mechanism for tokenization is Byte pair encoding from paper - https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf 
read the input representation part of the paper

- Tricky -> cause there are many languages and other symbols like special chars and emogi's

In [ ]:
[ord(x) for x in "안녕하세요 👋 (hello in Korean!)"]

: 

In [2]:
list("안녕하세요 👋 (hello in Korean!)".encode('utf8')) # utf-16 and utf-32 are very wasteful 

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

## Why not just use raw utf-8?

then we only have 0-265 vocab, so small encoding vector, but the same string will be represented in a much larger sequence then using chunks. which is not computationally or cost effective

## how byte pair encoding works?
In a given sequence - we iteratively find out the pairs that occurs frequently 
and represent then using a small symbolic repr (that decresase the sequence size but increases our 
vocab size) till we are satisfied

- see - https://en.wikipedia.org/wiki/Byte-pair_encoding

In [5]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode('utf8') # raw bytes
tokens = list(map(int, tokens)) # convert to a list of ints in range 0...255

len(tokens)

616

In [ ]:
# what we want to do, iterate over tokens and find the pair that occures most frequently, then we merge it
from collections import defaultdict

vocab_size = 276
num_merges = vocab_size - 256


def get_stats(tokens):
    pair_count = defaultdict(int)
    for i in range(1, len(tokens)):
        pair_count[(tokens[i-1], tokens[i])] += 1
    return pair_count

def merge(tokens, pair, idx):
    temp = []
    j = 0
    while j < len(tokens) :
        x, y = pair
        if j < len(tokens) - 1 and tokens[j] == x and tokens[j+1] == y:
            temp.append(idx)
            j += 1
        else:
            temp.append(tokens[j])
        j += 1
    return temp
    
def bpe(tokens, num_merges):
    mint = {}
    tokens = tokens
    nextt = 256
    for _ in range(num_merges):
        
        # find the count of pairs
        pair_count = get_stats(tokens)
        # breaks if not element or find pair wiht max coint
        if not pair_count:
            break
            
        pair = tuple(max(pair_count, key=pair_count.get))

        # replace the pair with mint
        tokens = merge(tokens, pair, nextt)
    
        # update the variables
        mint[pair] = nextt
        nextt += 1

    return mint, tokens

mint, encoded = bpe(tokens, num_merges)

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in mint.items():
    vocab[idx] = vocab[p0] + vocab[p1]

# def decode(tokens, mint):
#     out = tokens
#     mint = reversed(mint)
#     for m, pair in mint:
#         temp = []
#         for i in out:
#             # print(i)
#             if i == m:
#                 temp.extend(list(pair))
#             else:
#                 temp.append(i)
#         out = temp
#     return out

def decode(ids):
    #given ids (list of integers) return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf8", errors='repalce')
    return text

def encode(text):
    # given a string encode it into tokens
    tokens = list(text.encode('utf8'))
    while len(tokens) >= 2:
        stats = list(get_stats(tokens))
        pair = min(stats, key=lambda p: mint.get(p, float("inf")))
        if pair not in mint:
            break # nothing else can be merged
        idx = mint[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(decode(encode("Hello world!")))
print(decode(encode("h"))) # len(tokens) >= 2 for this
decoded = decode(encoded)

print(decoded == text)

: 

Note: not all words can be utf encoded for example, decoding some list of integer and then 
encoding them might not work the same. But for our purposes encoding the string and then decoding it will.

Now, let's see some state of the art tokenizers and see how are they differernt

# GPT - Tokenizer 

- https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
- https://github.com/openai/gpt-2/blob/master/src/encoder.py

Form paper:

We observed
BPE including many versions of common words like dog
since they occur in many variations such as dog. dog!
dog? . This results in a sub-optimal allocation of limited
vocabulary slots and model capacity. To avoid this, we prevent BPE from merging across character categories for any
byte sequence. We add an exception for spaces which significantly improves the compression efficiency while adding
only minimal fragmentation of words across multiple vocab
tokens.

This input representation allows us to combine the empirical
benefits of word-level LMs with the generality of byte-level
approaches. Since our approach can assign a probability to
any Unicode string, this allows us to evaluate our LMs on
any dataset regardless of pre-processing, tokenization, or
vocab size.

What they are essentially saying is:

# Forced splits using regex patterns

In [2]:
# from gpt-2 code on gh

import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello world"))  # finds all the matching patterns in a given string

['Hello', ' world']


so the above patters means:
- 's, 't, 're, 've, 'm, 'll, 'd 
- ?\p{L}+: words. A run of letters, optionally with one space in front. " hello", "café", " 世界".
- ?\p{N}+: numbers. A run of digits, optionally with one space in front. " 2026", "42".
- ?[^\s\p{L}\p{N}]+: punctuation and symbols. Anything that isn’t a space, letter, or digit, optionally with one space in front. "!!", " ->", " $", "(".
- \s+(?!\S): runs of whitespace between things, minus the last space (which gets handed to the next word/number/symbol). Catches indentation in code, double spaces, newlines. (For programming langs, like python)
- \s+: leftover whitespace, mainly trailing spaces at the very end of the text.

In [7]:
print(re.findall(gpt2pat, "Hello world123, 1. We'll start      python now!")) 

['Hello', ' world', '123', ',', ' 1', '.', ' We', "'ll", ' start', '     ', ' python', ' now', '!']


In [8]:
# then run BPE on this list but we don't mingle diff elements, then concatenate
# Still, this doesn't covers all the possible cases. like uppercases or unicode appostrophe

In [9]:
example = """
for i in range(1, 101):
    if i % 2 == 0:
        print('Even')
    else:
        print('Odd')
"""
print(re.findall(gpt2pat, example))

['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 2', ' ==', ' 0', ':', '\n       ', ' print', "('", 'Even', "')", '\n   ', ' else', ':', '\n       ', ' print', "('", 'Odd', "')", '\n']


In [11]:
import tiktoken

# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]


In [12]:
gpt4pat =  re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s""")

In [17]:
import os, json

with open('encoder.json', 'r') as f:
    encoder = json.load(f)

with open('vocab.bpe', 'r', encoding='utf8') as f:
    bpe_data = f.read()
bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]

## special tokens

In [20]:
len(encoder) # 256 raw byte toke + 50,000 merges + 1 special token

50257

In [25]:
encoder['<|endoftext|>'] # last token of GPT-2 tokenizers

50256

In [26]:
# special toekens are concatenated at at the end of merges from bpe

In [28]:
# create a minibpe tokenzation tool following the exercise on Andrej's gh

### sentencepiece

Commonly used because (unlike tiktoken) it can efficiently both train and inference BPE tokenizers. It is used in both Llama and Mistral series.

[sentencepiece on Github link](https://github.com/google/sentencepiece).

**The big difference**: sentencepiece runs BPE on the Unicode code points directly! It then has an option `character_coverage` for what to do with very very rare codepoints that appear very few times, and it either maps them onto an UNK token, or if `byte_fallback` is turned on, it encodes them with utf-8 and then encodes the raw bytes instead.

TLDR:

- tiktoken encodes to utf-8 and then BPEs bytes
- sentencepiece BPEs the code points and optionally falls back to utf-8 bytes for rare code points (rarity is determined by character_coverage hyperparameter), which then get translated to byte tokens.

(Personally I think the tiktoken way is a lot cleaner...)

In [32]:
import sentencepiece as spm

In [33]:
# write a toy.txt file with some random text
with open("toy.txt", "w", encoding="utf-8") as f:
  f.write("SentencePiece is an unsupervised text tokenizer and detokenizer mainly for Neural Network-based text generation systems where the vocabulary size is predetermined prior to the neural model training. SentencePiece implements subword units (e.g., byte-pair-encoding (BPE) [Sennrich et al.]) and unigram language model [Kudo.]) with the extension of direct training from raw sentences. SentencePiece allows us to make a purely end-to-end system that does not depend on language-specific pre/postprocessing.")

In [41]:
# train a sentencepiece model on it
# the settings here are (best effort) those used for training Lalma 2
import os

options = dict(
    # input spec
    input='toy.txt',
    input_format="text",
    # ouptput spec
    model_prefix="tok400", # output filename prefix
    # algo spec
    # bpe algo
    model_type="bpe",
    vocab_size=400,
    # normalization
    normalization_rule_name="identity", # turn off normalization
    remove_extra_whitespaces=False,
    input_sentence_size=200000000, # max num of training sentences
    max_sentence_length=4192, # max num of bytes per sentence
    seed_sentencepiece_size=1000000,
    shuffle_input_sentence=True,
    # rare work(code point) treatement
    character_coverage=0.99995,
    byte_fallback=True,
    # merge rules
    split_digits=True,
    split_by_unicode_script=True,
    split_by_whitespace=True,
    split_by_number=True,
    max_sentencepiece_length=16,
    add_dummy_prefix=True,
    allow_whitespace_only_pieces=True,
    # special tokens
    unk_id=0, # the UNK token MUST exist [unknown, mostly when byte fallback is False]
    bos_id=1, # the others are optional,set to -1 to turn off [begining of sentence]
    eos_id=2, # [end of sentence]
    pad_id=-1,
    # systems
    num_threads=os.cpu_count(), # use all system resources
)

spm.SentencePieceTrainer.train(**options)

True

In [42]:
sp = spm.SentencePieceProcessor()
sp.load('tok400.model')
vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
vocab

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
 ['<0x05>', 8],
 ['<0x06>', 9],
 ['<0x07>', 10],
 ['<0x08>', 11],
 ['<0x09>', 12],
 ['<0x0A>', 13],
 ['<0x0B>', 14],
 ['<0x0C>', 15],
 ['<0x0D>', 16],
 ['<0x0E>', 17],
 ['<0x0F>', 18],
 ['<0x10>', 19],
 ['<0x11>', 20],
 ['<0x12>', 21],
 ['<0x13>', 22],
 ['<0x14>', 23],
 ['<0x15>', 24],
 ['<0x16>', 25],
 ['<0x17>', 26],
 ['<0x18>', 27],
 ['<0x19>', 28],
 ['<0x1A>', 29],
 ['<0x1B>', 30],
 ['<0x1C>', 31],
 ['<0x1D>', 32],
 ['<0x1E>', 33],
 ['<0x1F>', 34],
 ['<0x20>', 35],
 ['<0x21>', 36],
 ['<0x22>', 37],
 ['<0x23>', 38],
 ['<0x24>', 39],
 ['<0x25>', 40],
 ['<0x26>', 41],
 ['<0x27>', 42],
 ['<0x28>', 43],
 ['<0x29>', 44],
 ['<0x2A>', 45],
 ['<0x2B>', 46],
 ['<0x2C>', 47],
 ['<0x2D>', 48],
 ['<0x2E>', 49],
 ['<0x2F>', 50],
 ['<0x30>', 51],
 ['<0x31>', 52],
 ['<0x32>', 53],
 ['<0x33>', 54],
 ['<0x34>', 55],
 ['<0x35>', 56],
 ['<0x36>', 57],
 ['<0x37>', 58],
 ['<0x38>', 5

In [43]:
ids = sp.encode("hello 안녕하세요")
print(ids)

[362, 378, 361, 372, 358, 362, 239, 152, 139, 238, 136, 152, 240, 152, 155, 239, 135, 187, 239, 157, 151]


In [44]:
print([sp.id_to_piece(idx) for idx in ids])

['▁', 'h', 'e', 'l', 'lo', '▁', '<0xEC>', '<0x95>', '<0x88>', '<0xEB>', '<0x85>', '<0x95>', '<0xED>', '<0x95>', '<0x98>', '<0xEC>', '<0x84>', '<0xB8>', '<0xEC>', '<0x9A>', '<0x94>']


### the code points not seen - korean chars, so what happens to these unknown tokens?

As byte fallback is true, spm falls back to bytes i.e. encode the unknown code points to utf-8 as use those as vocab repr


### Also worth looking at

- [Huggingface Tokenizer](https://huggingface.co/docs/transformers/main_classes/tokenizer). I didn't cover it in detail in the lecture because the algorithm (to my knowledge) is very similar to sentencepiece, but worth potentially evaluating for use in practice.

### Final recommendations

- Don't brush off tokenization. A lot of footguns and sharp edges here. Security issues. Safety issues.
- Eternal glory to anyone who can delete tokenization as a required step in LLMs.
- In your own application:
  - Maybe you can just re-use the GPT-4 tokens and tiktoken?
  - If you're training a vocab, ok to use BPE with sentencepiece. Careful with the million settings.
  - Switch to minbpe once it is as efficient as sentencepiece :)
